### Architecture

```
# config.json

num_hidden_layers   = 93
attn_res_block_size = 12
full_attn_layers    = [4, 8, 12, ..., 88, 92, 93]   # 24 个 MLA
kda_layers          = 69 个                 # 23 组 (3 KDA + 1 MLA) = 92 层，第 93 层再补一个 MLA
"num_heads": 96,
"qk_nope_head_dim": 128,
"num_experts": 896,
```

- 93 layers
    
```python
# config.json: `"num_hidden_layers": 93,`
# modeling_kimi_linear.py
class KimiLinearModel(KimiPreTrainedModel):
    self.layers = nn.ModuleList([KimiDecoderLayer(
        config, layer_idx) for layer_idx in range(config.num_hidden_layers)])
```

- $23\times4+1=93$ 与 $7\times12+9=93$ 是同一堆层的两种切法：
    - 前者切在注意力类型的周期上（无代码实体），
        - 4: 3KDA + 1MLA, 
    - 后者切在 AttnRes 的状态冻结点上（有代码实体）。12 = 3×4 让两者对齐，所以 AttnRes 块边界永远落在混合块边界上。

#### kimi decoder layer

- "each layer has two sub-layers": layer = Attention + FFN

```python
# 选择 ①：注意力类型            [modeling_kimi_linear.py:883-892]
if config.is_kda_layer(layer_idx):   self.self_attn = KimiDeltaAttention(...)
elif config.is_mla:                  self.self_attn = KimiMLAAttention(...)

# 选择 ②：FFN 类型              [modeling_kimi_linear.py:893-900]
if (config.num_experts is not None
    and layer_idx >= config.first_k_dense_replace     # = 1
    and layer_idx % config.moe_layer_freq == 0):      # = 1
    self.block_sparse_moe = KimiSparseMoeBlock(config)
else:
    self.mlp = KimiMLP(config)
```

### tensor shape aware

In [1]:
96 * 128, 12288 / 7168

(12288, 1.7142857142857142)

- KDA 的 num_heads=96、head_dim=128，所以 q/k/v_proj 都是 `[12288, 7168]`，KDA 内部宽度是 hidden 的 1.71 倍。

### KDA

> folding -> unfolding, roll -> unroll

$ q_t, k_t\in\mathbb R^{d_k}$（已 L2Norm，故 $| k_t|_2=1$），$ v_t\in\mathbb R^{d_v}$，状态 $\mathbf S_t\in\mathbb R^{d_k\times d_v}$，$\alpha_t\in(0,1)^{d_k}$，$\beta_t\in(0,1)$：

$$
S_t=\underbrace{\left(I-\beta_t k_t k_t^\top\right)\operatorname{Diag}(\alpha_t)}_{=:T_t\in\mathbb{R}^{d_k\times d_k}}S_{t-1}+\underbrace{\beta_t k_t v_t^\top}_{=:W_t},\qquad \tilde{o}_t=S_t^\top q_t.
$$

$\mathbf T_t$ 只从左边乘，只作用在 $d_k$ 轴上。$d_v$ 轴全程不参与转移，只是搭车。所以这是一条 $d_k$ 维的线性时变递推（linear time-varying recurrence），矩阵状态只是「$d_v$ 个独立的 $d_k$ 维系统并排放着」。

经典 KV 缓存存的是每个 token 各自的 $\mathbf k_s,\mathbf v_s$ 向量本身，沿 token 轴只追加、不修改，尺寸 $\propto T$；KDA 存的是这些 $(\mathbf k_s,\mathbf v_s)$ 对按累积衰减加权求和后的一块定长外积累加器（outer-product accumulator） $\mathbf S\in\mathbb R^{d_k\times d_v}$，沿 token 轴原地覆盖，尺寸与 $T$ 无关。二者不是两种东西，是同一份信息在结合律两种切法下的两种物化形态。

> $\mathbf S$ 就是 KV 缓存沿 token 轴收缩掉的结果

softmax 注意力读出 $\mathbf o_t=\sum_s \operatorname{softmax}_s(\mathbf q_t^\top\mathbf k_s),\mathbf v_s$。$\exp(\mathbf q^\top\mathbf k)$ 不能写成有限维的 $\varphi(\mathbf q)^\top\psi(\mathbf k)$，再加上分母跨 $s$ 耦合，求和无法与 $\mathbf q_t$ 分离——所以必须把每个 $\mathbf k_s,\mathbf v_s$ 原样留着，等 $\mathbf q_t$ 到了再算。

线性注意力把核换成恒等映射，读出变成：$o_t=\sum_{s\le t}(q_t^\top k_s)v_s=\left(\underbrace{\sum_{s\le t}k_sv_s^\top}_{S_t}\right)^\top q_t$


#### KDA & MLA

MLA 在这件事上和经典 MHA 没有任何区别，只是每 token 存的东西从 $2d=14,336$ 个数变成 $d_c+d_r=576$ 个数。逐 token 分页、逐块哈希、命中即共享，逐字照搬。真要说，它比 MHA 更适合细粒度——27 KiB/token 意味着 512 token 的哈希块才 13.5 MiB，切到 128 token 一块也毫无压力。

$\mathbf S_T$ 里恢复不出 $\mathbf S_B$（$\mathbf M_t=(\mathbf I-\beta_t\mathbf k_t\mathbf k_t^\top)\operatorname{Diag}(\mathbf\alpha_t)$ 一般不可逆）。所以对 KDA 而言：

全部来自 `config.json` 里 `linear_attn_config` 的三个数：`head_dim: 128`、`num_heads: 96`、`kda_layers` 列表长度 69。

| 层级 | 算式 | 结果 |
|---|---|---|
| 单个头的状态 | $d_k\times d_v=128\times128$ | 16,384 个数 |
| 一层（96 个头各一块方阵） | $\times\,96$ | 1,572,864 个数 |
| 全部 69 个 KDA 层 | $\times\,69$（共 6,624 块 $128^2$ 方阵） | 108,527,616 个数 |
| BF16，每个数 2 字节 | $\times\,2$ | **217,055,232 字节** |

$217\,055\,232/1024^2=\mathbf{207.0\ MiB}$，$/10^6=\mathbf{217.1\ MB}$——前面两处写的 217 MB 和 207 MiB 是同一个字节数，只是二进制单位与十进制单位的差别（$\times1.048576$）。

这就是代码里 `cache_params.recurrent_states` 那 69 个形状为 `(1, 96, 128, 128)` 的张量加起来的字节数，$B=1$。并发 32 个请求就是 6.5 GiB。

还要加上 `conv_states`：每层 3 条（q/k/v）× 12288 通道 × 窗口 4 = 147,456 个数，69 层 = 19.4 MiB，占 9.4%。**一份完整的可恢复状态 = 207.0 + 19.4 = 226.4 MiB。**（论文只写了 `persists the recurrent state`，但 ShortConv 的滑窗同样是跨 token 的状态，不一起存的话恢复后头 3 个 token 的 $q/k/v$ 会算错——这一条是我按正确性补的，不是论文原文。）

两个理解锚点：

- 它**与上下文长度完全无关**。1 个 token 的请求和 100 万 token 的请求，这块都是 226 MiB。
- 它等价于 **7,851 个 token 的 MLA 缓存**（$226.4\text{ MiB}\ /\ 27\text{ KiB per token}$）。这个数和 KDA/MLA 显存持平点是同一个，因为两者都是 $\frac{\text{一份定长状态}}{\text{每 token 的 MLA 开销}}$。

**checkpoint 就是那 226 MiB 字节的一份原样拷贝，外加一个标签："这是喂完 tokens[0:B) 之后的状态"。** 没有压缩、没有增量编码、没有变换，就是 `memcpy`。

一份 ckpt 226.4 MiB，而一个 512-token 哈希块的 MLA 缓存才 13.5 MiB——ckpt 是它的 16.8 倍。


走到 token 100,000 时，那块 226 MiB 里装的是 $\mathbf S_{100000}$。$\mathbf S_{60928}$ 已经**不存在了**——被 $\mathbf M_t$ 一层层碾过去，而 $\mathbf M_t=(\mathbf I-\beta_t\mathbf k_t\mathbf k_t^\top)\operatorname{Diag}(\mathbf\alpha_t)$ 不可逆，倒推不回来。想要以后还能从 60,928 接着算，唯一办法是**当时就把那 226 MiB 抄一份留下**。这份拷贝就叫 checkpoint。

关键对照，这句话说清楚就懂了：

> **MLA 的 KV 缓存本身就是它自己的 checkpoint。**

逐 token 追加意味着每一个 token 边界上都自动存在一份可恢复点，而且**免费**——那些字节本来就得存，取前 $B$ 行就等于"$B$ 处的存档"。KDA 用定长状态把这份逐 token 存储省掉了（每 token 0 字节），代价就是必须显式地、稀疏地把存档补回来。**省下的和补回的是同一笔账**，只是形态从"27 KiB × 每一个位置"变成了"226 MiB × 极少数位置"。

这笔账的汇率随上下文长度剧烈变化：

| 前缀长度 $T$ | 该段 MLA 缓存 | 一份 ckpt / 它 |
|---|---|---|
| 1,000 | 26.4 MiB | **8.59×**（存不起） |
| 8,000 | 210.9 MiB | 1.07× |
| 61,000 | 1.57 GiB | 0.14× |
| 1,048,576 | 27.0 GiB | 0.01×（几乎免费） |

所以策略只能是"稀疏地存"：论文的做法是每次 forward pass 之后在**本次处理到的最后一个哈希对齐位置**存一份，请求继续推进时前一份被取代就回收，只有落在**对话轮边界**上的保留下来供跨请求复用。decode 阶段每个 token 都在原地改 $\mathbf S$，但不逐 token 存 ckpt——所以**一次生成的中途状态是不可复用的**，只有那些刻意留下的边界可以。

另外：我上面把 69 层的状态合起来算作"一份 ckpt"。在引擎里它们分属若干个 KDA cache group、各自独立存放，但恢复请求需要**每个 group 都在**，所以淘汰时必须原子地同进同退——这就是前面那第 3 条一致性规则的由来。



> The intervening KDA layers provide position-sensitive and recency-aware sequence mixing, while the MLA layers provide unrestricted global content interaction.

LLaMA / Qwen / Mistral 这一系，每一层的每个头都在 $q^\top k$ 之前把 RoPE 打进去：

```python
q, k = apply_rotary_pos_emb(q, k, cos, sin)   # 每层都调
scores = q @ k.transpose(-2,-1) / sqrt(d)
```

RoPE 的性质是 $\langle R_m\mathbf q,\;R_n\mathbf k\rangle=\langle\mathbf q,\;R_{n-m}\mathbf k\rangle$——内积只依赖相对偏移 $n-m$。所以位置信息是**每层、每头、直接注入打分函数**的。cos/sin 表可以跨层共用，但施加动作是逐层的。

不只是 MLA 层没有，是**全模型零 rotary**：

```python
self.rotary_emb = None          # KimiMLAAttention.__init__:403
assert self.use_nope
```

KDA 那边本来就不用 RoPE。那 `qk_rope_head_dim = 64` 的通道呢？在 DeepSeek 的 MLA 里，这 64 维是**专门为承载 RoPE 而存在**的——因为 RoPE 和「把上投影吸收进 $q$」那个技巧不交换，必须留一路解耦的键不走压缩。K3 把 RoPE 关掉之后，这 64 维**结构保留、位置功能归零**，退化成一个头间共享（MQA 式）的键子空间。保留它更像是沿用 DeepSeek 的骨架，也让 KV cache 的 $576=512+64$ 布局不变。

> 位置信息从哪来：主要是 KDA，但不止「state 里存着」

KDA 提供的位置信息有**四个来源**，强度递减：

**(1) 累积衰减 $\mathbf\Gamma$ 是一个学出来的相对位置核。** 这是主力。把递推展开：

$$\tilde{\mathbf o}_t=\mathbf S_t^\top\mathbf q_t=\sum_{s\le t}\bigl(\cdots\bigr)\cdot\mathbf\Gamma^{s+1\to t},\qquad \mathbf\Gamma^{s+1\to t}=\prod_{r=s+1}^{t}\mathbf\alpha_r$$

$\mathbf\alpha$ 大致恒定时 $\mathbf\Gamma\approx\mathbf\alpha^{\,t-s}$——**距离 $t-s$ 的指数函数**，而且是逐通道、逐头各自学的。这在功能上就是一个 relative-position bias，只不过它乘在值上而不是加在 logits 上。（这正是 [kda-decay-unroll](onepagers/kda-decay-unroll.pdf) 那张图在讲的事。）

**(2) delta 规则让状态依赖顺序，而不只是距离。** 状态转移算子 $(\mathbf I-\beta_t\mathbf k_t\mathbf k_t^\top)\mathrm{Diag}(\mathbf\alpha_t)$ 之间**不可交换**，所以把两个 token 调换位置，$\mathbf S_t$ 真的会变——这比单纯的距离衰减更强。

**(3) ShortConv，kernel = 4。** KDA 的 q/k/v 每一路都先过一个宽度 4 的逐通道因果卷积。这是一个显式的局部位置算子，能区分 $t,t{-}1,t{-}2,t{-}3$。

**(4) 因果掩码本身。** 这条 MLA 层自己也有：即使完全 NoPE，因果注意力也不是置换不变的（可见 token 的数量随位置变化，文献里 decoder-only NoPE 能学到隐式位置）。但这个信号弱、外推差，K3 不靠它当主力。

所以准确的说法是：**MLA 的 $\mathbf q,\mathbf k$ 是从一条已经被 KDA 反复位置化过的残差流上算出来的**，位置信息通过**内容**进入 MLA，而不是通过打分函数。

### Stable LatentMoE


|        | gate | up | down | 类 |
|--------|------|----|------|-----|
| 路由专家 | w1 | w3 | w2 | KimiBlockSparseMLP |
| 共享专家 / 稠密 MLP | gate_proj | up_proj | down_proj | KimiMLP |

$$
h = \mathbf{W}_{\mathrm{down}}\!\left[\phi(\mathbf{W}_g x)\odot \mathbf{W}_u x\right]
$$

| | hidden | $d_{\mathrm{ff}}$ | $d_{\mathrm{ff}}/h$ | 参数 |
|---|---:|---:|---:|---:|
| 稠密 MLP（仅 layer 0） | 7168 | 33792 | $4.71\times$ | 727M |
| 共享专家（2 个合并成 1 个） | 7168 | 6144 | $6/7$ | 132M |
| 路由专家（$\times 896$） | 3584 | 3072 | $6/7$ | 33.0M |

- GLU：Gated Linear Unit

```python
gate_up = torch.cat([self.w1(x), self.w3(x)], dim=-1)   # 或 gate_proj / up_proj
h = self.w2(self.act_fn(gate_up))                        # 或 down_proj


# Register Moonshot-specific activation functions
class SituAndMul(nn.Module):
    """
    SituAndMul activation: beta * tanh(gate / beta) * sigmoid(gate) * up
    When linear_beta is set, up is also transformed by linear_beta * tanh(up / linear_beta).
    """
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        d = x.shape[-1] // 2
        gate = x[..., :d].to(torch.float32)
        up = x[..., d:].to(torch.float32)
        situ_a = self.beta * torch.tanh(gate / self.beta) * torch.sigmoid(gate)
        if self.linear_beta is not None:
            up = self.linear_beta * torch.tanh(up / self.linear_beta)
        return (situ_a * up).to(x.dtype)

```

### AttnRes

- $L=93,; S=12 \Rightarrow$ 前 7 个满 block（层 0–83）+ 最后 9 层的 partial block，共 8 个 block；再算上 embedding 作为 $ b_0$，K/V 集合最大 9 路。这正是 paper 说的 "8 blocks with 12-layer size, giving a partial final block and 9 total blocks when counting the embedding layer"。
- $ q_l =  w_l \in \mathbb R^{d}$ 是学出来的常量向量，不依赖 token（所以叫 pseudo-query，"伪"就伪在这里）；$\alpha$ 是对深度做 softmax，不是对序列位置。
    - Block AttnRes 把 $O(L)$ 路压成 $O(N)$ 路（:198-205）：块内先求和 $ b_n=\sum_{j\in\mathcal B_n} f_j( h_j)$，$ b_n^{i-1}$ 是块内前 $i-1$ 个 module 的部分和，$ b_0 =  h_1$（token embedding）。第 $n$ 块第 $i$ 个 module 的 value 矩阵是：

### reasoning effort

任务相对预算（per-problem budget）：

$$
T(y)>\tau b_0(x)
\quad\Longrightarrow\quad
R_{\text{task}}=-1
$$

- $x$：当前任务；
- $b_0(x)$：冷启动模型完成这个任务时估计出的初始预算；
- $T(y)$：当前轨迹实际消耗的 token；
- $\tau$：预算倍率。
- 先使用较大的 $\tau$ 训练 max 专家，再逐渐减小 $\tau$，得到 high 和 low 专家。因此预算是相对于任务难度的：复杂任务天然可以得到更大的绝对预算。

### MOPD

$$
r_{\mathrm{opd}}^{d}\!\left(y_t \mid e,x,y_{<t}\right)
=\operatorname{clip}\!\left(
\operatorname{sg}\!\left[
\log\frac{\pi_{\mathrm{teacher}}^{(d,e)}\!\left(y_t \mid x,y_{<t}\right)}{
\pi_{\theta}\!\left(y_t \mid e,x,y_{<t}\right)
}\right],-R_{\max},R_{\max}\right).
$$

- 忽略裁剪时，在学生分布上取期望：$\mathbb{E}_{y_t\sim\pi_\theta}\left[\log\pi_{\text{teacher}}(y_t)-\log\pi_\theta(y_t)\right]=-D_{\mathrm{KL}}\left(\pi_\theta\Vert\pi_{\text{teacher}}\right)$

### partial rollout

相对于当前模型，这些旧 token 就是陈旧数据（stale data）或离策略数据（off-policy data）。技术报告只说其策略优化包含 token 级正则化（per-token regularization），把每次更新约束在局部邻域内，从而容忍极端 off-policy 数据；但没有公开该正则项和完整 loss 的公式。

设第 $i$ 轮策略为 $\pi_{\theta_i}$，共有 $N$ 个 prompt，每个采样 $K$ 条轨迹。K3 的实际控制流是：
- 用 $\pi_{\theta_i}$ 启动 $NK$ 条轨迹。
- 等到其中 $\lambda NK$ 条完成。
- 暂停剩余的慢轨迹，保存模型前缀和 sandbox 状态。
- 用已完成的数据执行一次策略优化：$\theta_i\rightarrow\theta_{i+1}$

| | K3（co-located + partial rollout） | 典型异步 RL（AReaL / slime async 等） |
|---|---|---|
| GPU 部署 | rollout 与 train 分时复用同一批卡 | disaggregated，两组卡各司其职 |
| 时间关系 | 严格交替，互斥 | 真正重叠 |
| 权重同步 | 更新即时可见，无需 broadcast | 需显式 weight sync，且有版本管理 |
| staleness 形态 | 轨迹内部分段拼接，逐 token 递减 | 整条轨迹均匀陈旧 $m$ 步 |
| 长尾如何消化 | $\lambda$ 分位抢占 + 跨迭代续跑 | rollout 侧自然吸收，训练不等 |
| 成本 | 几百卡即可跑 1M context | 需为两侧各配一整套能装下模型的卡 |